In [1]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd


In [2]:
DATA_DIR = Path("../../data/algemeen/gebeurtenisdata_macro")

excel_files = sorted(f for f in DATA_DIR.glob("*.xlsx") if not f.name.startswith(("._", "~$")))

if not excel_files:
    raise FileNotFoundError(
        f"No Excel files were found in '{DATA_DIR}'. "
        "The research data are not included in this repository."
    )

dataframes = []

for file_path in excel_files:
    try:
        frame = pd.read_excel(file_path)
        dataframes.append(frame)
        print(f"Loaded: {file_path.name}")
    except Exception as exc:
        raise RuntimeError(
            f"Could not read '{file_path.name}'."
        ) from exc

df = pd.concat(dataframes, ignore_index=True)

print(f"Number of rows: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")

Loaded: 1253#4.xlsx


Loaded: 1699#126.xlsx


Loaded: 1700#10.xlsx


Loaded: 1700#11.xlsx


Loaded: 1700#13.xlsx


Loaded: 1700#15.xlsx


... (185 outputs left out)

Loaded: MA#33313.xlsx


Loaded: MA#33316.xlsx


Loaded: MA#41581.xlsx


Number of rows: 383,125
Number of columns: 42


In [3]:
df.columns

Index(['bron', 'archive_file', 'year', 'district', 'Person ID',
       'Incident Number', 'Page Number', 'E_DATE', 'E_TIME', 'E_LOC',
       'P_LASTNAME', 'P_FIRSTNAME', 'P_PROF', 'P_BIRTHPLACE', 'P_BIRTHDATE',
       'P_RES', 'P_TITLE', 'P_MARITALSTATUS', 'P_AGE', 'E_DATE_clean',
       'E_TIME_clean', 'E_TIME_grouped', 'E_LOC_gis1898',
       'firstname_firstword', 'gender_inferred', 'firstname_match',
       'P_PROF_match', 'P_PROF_hisco', 'P_PROF_hisclass', 'E_YEAR',
       'P_AGE_clean', 'P_BIRTHYEAR', 'P_AGE_categories', 'P_BIRTHPLACE_extra',
       'P_BIRTHPLACE_cleaned', 'P_BIRTHPLACE_match', 'P_BIRTHPLACE_country',
       'P_BIRTHPLACE_dis', 'P_BIRTHPLACE_sim', 'astro', 'incident_text',
       'E_LOC_gis1898_oud'],
      dtype='object')

In [4]:
# typering
df["year"]            = pd.to_numeric(df["year"], errors="coerce")
df["P_AGE_clean"]     = pd.to_numeric(df["P_AGE_clean"], errors="coerce")
df["gender_inferred"] = df["gender_inferred"].str.strip().str.lower()

In [5]:
KOL_KL = "P_PROF_hisclass"

print("waarden P_PROF_hisclass:")
print(df[KOL_KL].value_counts(dropna=False).head(20))
print(f"\ndekking totaal: {df[KOL_KL].notna().mean():.1%}")

d_kl = df[df["gender_inferred"].isin(["man", "woman"])
          & df["year"].between(1877, 1939)].copy()

print("\ndekking per gender:")
print(d_kl.groupby("gender_inferred")[KOL_KL].apply(lambda s: s.notna().mean()).round(3))
print("\ndekking per leeftijdsgroep:")
print(d_kl.groupby("P_AGE_categories")[KOL_KL].apply(lambda s: s.notna().mean()).round(3))

waarden P_PROF_hisclass:
P_PROF_hisclass
NaN        295814
9           21602
unknown     15130
11          13297
7           10662
4            9477
3            4854
2            3836
1            3172
5            2783
8            1348
6             779
12            315
10             56
Name: count, dtype: int64

dekking totaal: 22.8%



dekking per gender:
gender_inferred
man      0.290
woman    0.195
Name: P_PROF_hisclass, dtype: float64

dekking per leeftijdsgroep:
P_AGE_categories
jong kind         0.000
jongvolwassene    0.482
kind              0.161
oudere            0.368
volwassene        0.559
Name: P_PROF_hisclass, dtype: float64


In [6]:
HISCLASS3 = {1: "elite", 2: "elite",
             3: "middenklasse", 4: "middenklasse", 5: "middenklasse",
             6: "arbeiders", 7: "arbeiders", 8: "arbeiders",
             9: "arbeiders", 10: "arbeiders", 11: "arbeiders", 12: "arbeiders"}

# HISCLASS-code als getal
d_kl["hc"] = (pd.to_numeric(d_kl[KOL_KL].replace("unknown", np.nan), errors="coerce")
                .astype("Int64"))

# Correctie: herbergiers (HISCO 51020) -> middenklasse (klasse 5)
hisco_num = pd.to_numeric(d_kl["P_PROF_hisco"], errors="coerce")
d_kl.loc[hisco_num == 51020, "hc"] = 5
print(f"herbergiers hergecodeerd naar middenklasse: {(hisco_num == 51020).sum()}")

d_kl["klasse3"] = d_kl["hc"].map(HISCLASS3)

print(f"\nberoep bekend: {d_kl['klasse3'].notna().sum():,} van {len(d_kl):,} "
      f"({d_kl['klasse3'].notna().mean():.1%})")

print("\naandeel per gender (van wie beroep bekend is):")
print(pd.crosstab(d_kl["klasse3"], d_kl["gender_inferred"], normalize="columns").round(3))

d_jv = d_kl[d_kl["P_AGE_clean"].between(15, 24)]
print("\njongvolwassenen 15-24:")
print(pd.crosstab(d_jv["klasse3"], d_jv["gender_inferred"], normalize="columns").round(3))

print("\ntop HISCLASS-codes bij jonge vrouwen:")
print(d_jv[d_jv["gender_inferred"] == "woman"]["hc"].value_counts().head(6))

herbergiers hergecodeerd naar middenklasse: 1727

beroep bekend: 47,197 van 211,234 (22.3%)

aandeel per gender (van wie beroep bekend is):
gender_inferred    man  woman
klasse3                      
arbeiders        0.702  0.736
elite            0.029  0.015
middenklasse     0.269  0.248

jongvolwassenen 15-24:


gender_inferred    man  woman
klasse3                      
arbeiders        0.833  0.877
elite            0.006  0.003
middenklasse     0.161  0.120

top HISCLASS-codes bij jonge vrouwen:
hc
11    967
9     232
5     101
7      97
4      63
6      42
Name: count, dtype: Int64


/var/folders/m7/jhz7yh0x2rs_1grxt7m964600000gn/T/ipykernel_25631/2179391930.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d_kl["hc"] = (pd.to_numeric(d_kl[KOL_KL].replace("unknown", np.nan), errors="coerce")
